# Fine-tuning Language Models for Sequence Classification
This notebook explores the fine-tuning process of language models for sequence classification tasks, like sentiment analysis. We focus on textual data. We need to have the following libraries installed:

* `numpy`: needed for handly functions, like computing the _argmax_.
* `torch`: needed to handle GPU-related stuff.
* `scikit-learn`: needed to compute metrics to evaluate the model.
* `transformers`: needed to load the base model and tokenizer, and to handle all training-related classes.
* `datasets`: needed to load the datasets from HuggingFace, also useful for efficient training.
* `matplotlib`: needed to plot the confusion matrix at the end of training. You can comment this out if you don't need to do it.

Both `transformers` and `datasets` are libraries developed by HuggingFace 🤗, and therefore well integrated, saving us the need of cumbersome preprocessing.

> Before we start, we need to note that this code implements standard (full) fine-tuning, i.e., it fine-tunes all the parameters of the model. This can work well for several use-cases, but there are some potentioal problems that we need to keep in mind. Specifically, the need for more computational resources compared to parameter-efficient fine-tuning (PEFT) methods, which only fine-tune certain layers or lower-dimensional adapters.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## Configs
Here, we can set a few configuration parameters, like the model's path and training-related hyperparameters.

Regarding **model choice**, basically any language model can be used for sequence classification, as long as it is loaded using the `transformers.AutoModelForSequenceClassification` class. Even generative Large Language Models (LLMs) can be used as classifiers, as the generative head gets swapped for a classification head, that outputs a vector of the same dimension of the number of unique classes.

In [ ]:
MODEL_ID         : str   = 'FacebookAI/xlm-roberta-base' # path to model's Hugging Face repository
DATASET_ID       : str   = 'istat-ai/sentipolc_dataset'  # path to dataset's Hugging Face repository

OUTPUT_DIR       : str   = f'saved_models/your_ft_model' # output directory for the saved model
NUM_EPOCHS       : int   = 2                             # number of training epochs
LEARNING_RATE    : float = 2e-5                          # learning rate for weight updates
LR_SCHEDULER     : str   = 'linear'                      # learning rate decay scheduler
OPTIMIZER        : str   = 'adamw_torch_fused'           # optimizer to use for training
TRAIN_BATCH_SIZE : int   = 8                             # batch size during training (affects training)
EVAL_BATCH_SIZE  : int   = 16                            # batch size during evaluation (does not affect training)
GA_STEPS         : int   = 2                             # gradient accumulation steps, simulates higher batch size
WARMUP_RATIO     : float = 0.1                           # % of steps during which the lr increases before reaching the specified value
WEIGHT_DECAY     : float = 0.01                          # penalty for large weight values, prevents overfitting
LOGGING_STEPS    : int   = 100                           # how often training metrics are logged (steps)
EVAL_STEPS       : int   = 100                           # how often the model is evaluated (steps)
EVAL_STRATEGY    : str   = 'steps'                       # based on what the model is evaluated
SAVE_STRATEGY    : str   = 'steps'                       # based on what the model is saved
FP16             : bool  = True                          # mixed-precision training (use with older hardware, like T4 GPUs)
BF16             : bool  = False                         # mixed-precision training (use with hardware that supports Ampere+)
LOAD_BEST        : bool  = True                          # load the best performing model at end based on specified metric
REPORT_TO        : list  = []                            # which logging integrations to use (ex. tensorboard)
LOG_LEVEL        : str   = 'warning'                     # controls logging verbosity

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

<hr>

## Dataset

Let's load the dataset from HuggingFace. Your data needs to be a `datasets.DatasetDict` with the following structure to work with this notebook:

```
DatasetDict({
    train: Dataset({
        features: ['text', 'labels']
    })
    eval: Dataset({
        features: ['text', 'labels']
    })
    test: Dataset({
        features: ['text', 'labels']
    })
})
```
A few things:
* The `test` split is optional, it will be used only at the end if you want to cross-validate your model on unseen data.
* The `eval` split is also optional, but we _highly_ recommend providing it to prevent overfitting.
* The `labels` must be integrers. If they are another type, like strings, make sure to preprocess them and encode them into integers.
* If you have any additional features, it's ok, they will just be discarded automatically by the `Trainer`.

We will also extract the number of unique labels from the `train` split.

In [ ]:
data = load_dataset(DATASET_ID)
n_labels = len(data['train'].unique('labels'))

### Tokenization
Now we need to tokenize the data, i.e., encode the text into integers representing token IDs. To do this, we will use the pre-trained tokenizer associated to the model we want to fine-tune. Some models, like Llama LLMs, lack padding token IDs in the tokenizers, therefore we need to manually assign them (we will assign it to the EOS token ID, i.e., the end-of-sentence token). We need a padding token ID to train the model using batches of data, where each batch contains sequences of the same length.

First, let's load the tokenizer and extract the maximum sequence length that the model can process. Then, we can define a tokenize function and apply it to the text feature. The tokenize function will return the following features, in addition to the existing ones:

* `input_ids`: the tokenized sequences.
* `attention_mask`: the tensor used to mask the padding tokens.

These features, in addition to the `labels` feature (**important**: it needs to be exactly named "labels"!), are all we need to train the model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
max_len = tokenizer.model_max_length

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

def tokenize(example, max_len: int):
    return tokenizer(example['text'], padding=True, truncation=True, max_length=max_len, return_tensors='pt')

tokenized_data = data.map(lambda example: tokenize(example, max_len), batched=True)

Even if the `transformers.Trainer` automatically discards irrelevant features in the dataset, it is best practice to clean up the dataset before training. Therefore, we will remove each feature that is not in `['input_ids', 'attention_mask', 'labels']`.

Moreover, we need to make sure that every column is cast correctly as a PyTorch Tensor.

In [ ]:
relevant_features = ['input_ids', 'attention_mask', 'labels']

for split in tokenized_data.column_names:
    for feature in tokenized_data[split].column_names:
        if feature not in relevant_features:
            tokenized_data[split] = tokenized_data[split].remove_columns(feature)

tokenized_data.set_format(type='torch', columns=relevant_features)

<hr>

## Training
First, we load the base (pre-trained) model that we want to fine-tune. The same considerations about the padding token ID of the tokenizer apply here.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels = n_labels
).to(device)

if model.config.pad_token_id is None:
    model.config.pad_token_id = model.config.eos_token_id

Now we need to define a function to compute metrics during training. Usually F1 score metrics work well for multi-class classification problems, but you can add other metrics such as precision and recall. Just remember to import them from `sklearn.metrics`.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': accuracy, 'f1_macro': f1}

Finally, let's define the `TrainingArguments` and `Trainer` classes. This is the heart of the training process and where the main hyperparameters for training are set. They are wrappers for much more complicated procedures and they save us quite some time. We will also initialize a `DataCollatorWithPadding` class, which will help us in dynamically handling training batches.

In order to assess which model performed best during training, we need to define a metric for evaluation. We suggest to leave it as `'eval_loss'`, but you can also use any of the metrics retured from the `compute_metrics` function. If `'LOAD_BEST'` is set to `True`, the trainer will automatically load the weights of the best performing model at the end of training.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    optim=OPTIMIZER,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GA_STEPS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_dir='./logs',
    logging_steps=LOGGING_STEPS,
    metric_for_best_model='eval_loss',
    eval_steps=EVAL_STEPS,
    eval_strategy=EVAL_STRATEGY,
    save_strategy=SAVE_STRATEGY,
    fp16=FP16,
    bf16=BF16,
    load_best_model_at_end=LOAD_BEST,
    report_to=REPORT_TO,
    log_level=LOG_LEVEL,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data['train'],
    eval_dataset=tokenized_data['eval'] if 'eval' in tokenized_data else None,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

Let's fine-tune our model!

In [ ]:
trainer.train()

### Cross-validation
If a `'test'` split is provided in the dataset, we can perform cross-validation on previously unseen data. To do this, we just need to use the method `.evaluate()` of our trainer and specify the test dataset.

In [ ]:
if 'test' in tokenized_data:
    print(trainer.evaluate(eval_dataset=tokenized_data['test']))

We can also plot a confusion matrix to assess the performance on the different labels.

In [ ]:
if 'test' in tokenized_data:
    output = trainer.predict(tokenized_data['test'])

    logits = output.predictions
    predictions = np.argmax(logits, axis=-1)

    cm = confusion_matrix(tokenized_data['test']['labels'], predictions)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm)

    disp.plot()
    plt.show()

<hr>

## Export
As a final step, we could export the model to a HuggingFace repository. We need to have a Hugging Face account and provide a `HF_TOKEN` (either as an environment variable, named _exactly_ like that, or passing the token in the `token` argument of the `push_to_hub` function) to be able to do this. After this, we will be able to load our model from the hub and use it everywhere.

If you want to keep it in a private repository, set `PRIVATE = True`. Else, the model will be available for everyone to download.

In [ ]:
REPO_ID : str  = 'your-username/your-model-name'
PRIVATE : bool = True

trainer.push_to_hub(REPO_ID, private=PRIVATE)